# Linear models

An overview of linear and logistic regression models.

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from geodatasets import get_path
from sklearn import metrics

from spml.linear_model import GWLinearRegression, GWLogisticRegression

## Linear regression

Linear regression is a classical example of geographically weighted regression models. In this example, you can predict the number of suicides based on other population data in the [Guerry](https://geodacenter.github.io/data-and-lab//Guerry/) dataset.

In [ ]:
gdf = gpd.read_file(get_path("geoda.guerry"))

It is a relatively small dataset covering Frech data from 1800s.

In [ ]:
gdf.plot().set_axis_off()

Specify the model and fit the data.

In [ ]:
adaptive = GWLinearRegression(bandwidth=25, fixed=False, kernel="tricube")
adaptive.fit(
    gdf[["Crm_prp", "Litercy", "Donatns", "Lottery"]],
    gdf["Suicids"],
    geometry=gdf.representative_point(),
)

You can get a number of outputs from the fitted model. For example, focal predictions (a value is predicted on an observations using the local model fitted around it).

In [ ]:
adaptive.pred_

This allows you to measure model performance metrics like $R^2$. This value would be comparable to the R2 reported by `mgwr`.

In [ ]:
metrics.r2_score(gdf["Suicids"], adaptive.pred_)

Local version of $R^2$ is accessible directly.

In [ ]:
adaptive.local_r2_

Any of the values linked to individual observations can then be mapped.

In [ ]:
gdf.plot(adaptive.local_r2_, legend=True).set_axis_off()

Similarly, you can access residuals.

In [ ]:
gdf.plot(adaptive.resid_, legend=True).set_axis_off()

Each local model then have its own local coefficients.

In [ ]:
adaptive.local_coef_

Again, this can be explored visually.

In [ ]:
f, axs = plt.subplots(2, 2, figsize=(12, 10))

for column, ax in zip(adaptive.local_coef_.columns, axs.flat, strict=False):
    gdf.plot(adaptive.local_coef_[column], legend=True, ax=ax)
    ax.set_title(column)
    ax.set_axis_off()

Alongside local coefficients, you can retireve a local intercept value.

In [ ]:
gdf.plot(adaptive.local_intercept_, legend=True).set_axis_off()

For details on prediction, see the [Prediction](./predict.ipynb) guide.

## Logistic regression

`GWLogisticRegression` supports binary and multiclass classification, with boolean, numeric or string class labels. However, be aware that local neighborhoods may contain only a subset of the global classes. Fitted models report zero probability for absent classes; neighborhoods that are skipped report NaN for every class.

Let's first illustrate binary classification by predicting whether the number of suicides is over or under the median.

In [ ]:
y = gdf["Suicids"] > gdf["Suicids"].median()

The API then looks the same, following scikit-learn's model.

In [ ]:
binary = GWLogisticRegression(bandwidth=25, fixed=False, max_iter=500)
binary.fit(
    gdf[["Crm_prp", "Litercy", "Donatns", "Lottery"]],
    y,
    geometry=gdf.representative_point(),
)

Given this is a classification task, we can retrieve probabilites from focal predictions.

In [ ]:
binary.proba_

Or their binary counterparts (based on the maximum value).

In [ ]:
binary.pred_

The regions with missing values have no fitted local model. By default, a neighborhood is skipped if its target is invariant or if the ratio of its least frequent to most frequent class is below `min_proportion`. For example, counts of 2 and 10 meet the threshold, although the minority class represents less than 20% of all observations. In multiclass models, this ratio uses only classes present in the neighborhood.

Skipped models are unavailable for prediction, so some locations may have no local prediction. You can check the proportion of fitted local models using `prediction_rate_`.

In [ ]:
binary.prediction_rate_

This shows that 87% of focal geometries have their respective models, the rest does not.

For more on dealing with class imbalance, see the [imbalance guide](imbalance.ipynb).

The focal metrics (based on a prediction on a focal geometry using a single local model fitted around that geometry) can be measured using the two outputs shown above.

In [ ]:
na_mask = binary.pred_.notna()

metrics.accuracy_score(y[na_mask], binary.pred_[na_mask])

However, you can also extract all data from all local models and use those to measure the performance of the model, rather than using only focal geometries.

The data pooled from all local models are accessible as `y_pooled_` and `pred_pooled_`.

In [ ]:
metrics.accuracy_score(binary.y_pooled_, binary.pred_pooled_)

Pooled metrics are typically showing worse results as the distance decay of local observation weights in individual models are playing against a good prediction of values far from the focal point.

Alternatively, you can measure performance metrics per each local model. That is relying on the same pooled data but split to their respective parent local models.

In [ ]:
local_accuracy = binary.local_metric(metrics.accuracy_score)
local_accuracy

In [ ]:
gdf.plot(
    local_accuracy, legend=True, missing_kwds=dict(color="lightgray")
).set_axis_off()

For this binary target, `local_coef_` has one column per feature, as in the regression counterpart, and `local_intercept_` is a Series indexed by location.

In [ ]:
binary.local_coef_

Again, this can be explored visually.

In [ ]:
f, axs = plt.subplots(2, 2, figsize=(12, 10))

for column, ax in zip(binary.local_coef_.columns, axs.flat, strict=False):
    gdf.plot(
        binary.local_coef_[column],
        legend=True,
        ax=ax,
        missing_kwds=dict(color="lightgray"),
    )
    ax.set_title(column)
    ax.set_axis_off()

### Multiclass outputs

For a multiclass target, `local_coef_` has a `(class, feature)` column MultiIndex and `local_intercept_` is a DataFrame with one column per global class. Both have one row per focal location. Coefficients and intercepts for locally absent classes are NaN, since no parameters were estimated for them; skipped models have NaN for all classes.

For example, divide the target into three groups to simulate multiclass problem:



In [ ]:
X_multi = gdf[["Crm_prp", "Litercy", "Donatns", "Lottery"]]
X_multi = (X_multi - X_multi.mean()) / X_multi.std()

y_multi = pd.qcut(gdf["Suicids"], 3, labels=["low", "medium", "high"])

multiclass = GWLogisticRegression(bandwidth=25, max_iter=500)
multiclass.fit(X_multi, y_multi, geometry=gdf.representative_point())

multiclass.local_coef_

Local interctept is also reported as a DataFrame.

In [ ]:
multiclass.local_intercept_["high"]

We can further check the presence of individual classes in local neighbohoods.

In [ ]:
multiclass.local_class_presence_


When the global target is multiclass but a local neighborhood contains only two classes, the reported coefficients use symmetric logits: `-coef / 2` for the first local class and `coef / 2` for the second, with the same transformation for the intercept. This expresses the binary model with one parameter vector per class while preserving its probabilities under softmax. Prediction uses the original fitted estimator. Globally binary models retain the original coefficient representation.

Hat values, effective degrees of freedom, pooled log-likelihood and information criteria (`aic_`, `aicc_`, `bic_`) are currently exposed only for binary logistic models. For multiclass bandwidth selection, use log loss; see the [bandwidth search guide](bandwidth_search.ipynb).

See the [imbalance guide](imbalance.ipynb) for class presence and warning behavior, and the [Prediction](predict.ipynb) guide for prediction on new data.